In [64]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from yreport import data_health_report

In [65]:
movies = pd.read_csv('data/tmdb_5000_movies.csv')
credits = pd.read_csv('data/tmdb_5000_credits.csv')

In [66]:
health = data_health_report(movies).summary()

Data Health Score: 82.56/100
Rows: 4803 | No_Columns: 20

Numeric Columns : ['budget', 'id', 'popularity', 'revenue', 'runtime', 'vote_average', 'vote_count']
Categorical Columns : ['genres', 'homepage', 'keywords', 'original_language', 'original_title', 'overview', 'production_companies', 'production_countries', 'release_date', 'spoken_languages', 'status', 'tagline', 'title']
DateTime Columns : []

Missing Percentage: 
- homepage: 64.36%
- overview: 0.06%
- release_date: 0.02%
- runtime: 0.04%
- tagline: 17.57%

Warnings:
- high_missing: ['homepage']
- high_cardinality: ['genres', 'keywords', 'original_title', 'overview', 'production_companies', 'production_countries', 'release_date', 'spoken_languages', 'tagline', 'title']

Recommendations:
- encoding: {'genres': {'action': 'required', 'message': 'Categorical encoding required (high cardinality)', 'confidence': 'HIGH'}, 'keywords': {'action': 'required', 'message': 'Categorical encoding required (high cardinality)', 'confidence': 'H

In [67]:
movies.sample()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
1232,40000000,"[{""id"": 35, ""name"": ""Comedy""}, {""id"": 18, ""nam...",http://www.morningglorymovie.com/,38357,"[{""id"": 13281, ""name"": ""work ethic""}, {""id"": 1...",en,Morning Glory,A young and devoted morning television produce...,22.491388,"[{""name"": ""Bad Robot"", ""id"": 11461}, {""name"": ...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2010-01-12,58785180,102.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Breakfast TV just got interesting.,Morning Glory,6.1,412


In [68]:
credits.sample()

,movie_id,title,cast,crew
2482,12120,My Stepmother is an Alien,"[{""cast_id"": 10, ""character"": ""Steven Mills"", ...","[{""credit_id"": ""52fe44b99251416c7503ec53"", ""de..."


In [69]:
movies = movies.merge(credits,on='title')

In [70]:
movies.info()

<class 'pandas.DataFrame'>
RangeIndex: 4809 entries, 0 to 4808
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4809 non-null   int64  
 1   genres                4809 non-null   str    
 2   homepage              1713 non-null   str    
 3   id                    4809 non-null   int64  
 4   keywords              4809 non-null   str    
 5   original_language     4809 non-null   str    
 6   original_title        4809 non-null   str    
 7   overview              4806 non-null   str    
 8   popularity            4809 non-null   float64
 9   production_companies  4809 non-null   str    
 10  production_countries  4809 non-null   str    
 11  release_date          4808 non-null   str    
 12  revenue               4809 non-null   int64  
 13  runtime               4807 non-null   float64
 14  spoken_languages      4809 non-null   str    
 15  status                4809 non-n

### which column we keep
- genres
- movie_id
- keywords
- title
- overview
- cast
- crew

In [71]:
movies = movies[['movie_id','title','overview','genres','keywords','cast','crew']]

In [72]:
movies.sample()

,movie_id,title,overview,genres,keywords,cast,crew
12,58,Pirates of the Caribbean: Dead Man's Chest,Captain Jack Sparrow works his way out of a bl...,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 616, ""name"": ""witch""}, {""id"": 663, ""na...","[{""cast_id"": 37, ""character"": ""Captain Jack Sp...","[{""credit_id"": ""52fe4211c3a36847f8001873"", ""de..."


In [73]:
movies.isnull().sum()

movie_id    0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64

In [74]:
movies.dropna(inplace=True)

In [75]:
movies.isnull().sum()

movie_id    0
title       0
overview    0
genres      0
keywords    0
cast        0
crew        0
dtype: int64

In [76]:
movies.duplicated().sum()

np.int64(0)

In [77]:
movies['genres'][0]

'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

#### clean data
- [Action,Adventure,Fantasy,Science Fiction] in this format


In [78]:
import json
def convert(obj):
    result = []
    for movie in obj:
        movie = json.loads(movie)
        t_result = []
        for m in movie:
            t_result.append(m['name'])
        result.append(t_result)
    return result

In [79]:
movies['genres'] = convert(movies['genres'])

In [80]:
movies['genres'].head()

0    [Action, Adventure, Fantasy, Science Fiction]
1                     [Adventure, Fantasy, Action]
2                       [Action, Adventure, Crime]
3                 [Action, Crime, Drama, Thriller]
4             [Action, Adventure, Science Fiction]
Name: genres, dtype: object

In [81]:
movies['keywords'] = convert(movies['keywords'])

In [82]:
movies['keywords'].head()

0    [culture clash, future, space war, space colon...
1    [ocean, drug abuse, exotic island, east india ...
2    [spy, based on novel, secret agent, sequel, mi...
3    [dc comics, crime fighter, terrorist, secret i...
4    [based on novel, mars, medallion, space travel...
Name: keywords, dtype: object

In [83]:
import ast
def convert3(obj):
    result = []
    counter = 0
    for movie in ast.literal_eval(obj):
        if counter != 3:
            result.append(movie['name'])
            counter += 1
        else:
            break
    return result

In [84]:
movies['cast'] = movies['cast'].apply(convert3)

In [85]:
movies.head()

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[Johnny Depp, Orlando Bloom, Keira Knightley]","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...","[Daniel Craig, Christoph Waltz, Léa Seydoux]","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i...","[Christian Bale, Michael Caine, Gary Oldman]","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel...","[Taylor Kitsch, Lynn Collins, Samantha Morton]","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [86]:
def fetch_director(obj):
    result = []
    for movie in ast.literal_eval(obj):
        if movie['job'] == 'Director':
            result.append(movie['name'])
            break
    return result

In [87]:
movies['crew'] = movies['crew'].apply(fetch_director)

In [88]:
movies.head()

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron]
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[Johnny Depp, Orlando Bloom, Keira Knightley]",[Gore Verbinski]
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...","[Daniel Craig, Christoph Waltz, Léa Seydoux]",[Sam Mendes]
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i...","[Christian Bale, Michael Caine, Gary Oldman]",[Christopher Nolan]
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel...","[Taylor Kitsch, Lynn Collins, Samantha Morton]",[Andrew Stanton]


In [89]:
movies['overview'] = movies['overview'].apply(lambda x: x.split())
movies.head()

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron]
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[Johnny Depp, Orlando Bloom, Keira Knightley]",[Gore Verbinski]
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...","[Daniel Craig, Christoph Waltz, Léa Seydoux]",[Sam Mendes]
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i...","[Christian Bale, Michael Caine, Gary Oldman]",[Christopher Nolan]
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel...","[Taylor Kitsch, Lynn Collins, Samantha Morton]",[Andrew Stanton]


In [90]:
movies['genres'] = movies['genres'].apply(lambda x:[i.replace(' ','') for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x:[i.replace(' ','') for i in x])
movies['cast'] = movies['cast'].apply(lambda x:[i.replace(' ','') for i in x])
movies['cast'] = movies['cast'].apply(lambda x:[i.replace(' ','') for i in x])
movies['overview'] = movies['overview'].apply(lambda x:[i.replace(' ','') for i in x])

In [91]:
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [92]:
new_df = movies[['movie_id','title','tags']]

In [93]:
new_df['tags'] = new_df['tags'].apply(lambda x: ' '.join(x))
new_df['tags'] = new_df['tags'].apply(lambda x:x.lower())

In [94]:
new_df.head()

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...
3,49026,The Dark Knight Rises,following the death of district attorney harve...
4,49529,John Carter,"john carter is a war-weary, former military ca..."


In [95]:
new_df.shape

(4806, 3)

In [96]:
from nltk.stem.porter import PorterStemmer
stemmer = PorterStemmer()
def stem(text):
    y = []
    for word in text.split():
        word = stemmer.stem(word)
        y.append(word)
    return ' '.join(y)

In [97]:
new_df['tags'] = new_df['tags'].apply(stem)

## Vectorzation

In [98]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

cv = TfidfVectorizer(max_features=5000,stop_words='english')
vectors = cv.fit_transform(new_df['tags']).toarray()

In [99]:
vectors[0]

array([0., 0., 0., ..., 0., 0., 0.], shape=(5000,))

In [100]:
cv.get_feature_names_out()

array(['000', '007', '10', ..., 'zooeydeschanel', 'zucker', 'zwick'],
      shape=(5000,), dtype=object)

## Cosine similarity

In [101]:
from sklearn.metrics.pairwise import cosine_similarity
similarity = cosine_similarity(vectors)

In [102]:
similarity.shape

(4806, 4806)

In [103]:
similarity[0]

array([1.        , 0.02225596, 0.03071299, ..., 0.02375837, 0.        ,
       0.        ], shape=(4806,))

## Recommendation

In [104]:
def recommend(movie):
    index = int(new_df[new_df['title'] == movie].index[0])
    distances = similarity[index]
    movies_list = list(sorted(enumerate(distances), key=lambda x: x[1], reverse=True))[1:6]
    for movie in movies_list:
        print(new_df.iloc[movie[0]]['title'])
    return None

In [105]:
list(sorted(enumerate(similarity[0]), key=lambda x: x[1], reverse=True))[1:6]

[(2405, np.float64(0.26136418952889423)),
 (3728, np.float64(0.2308660503971189)),
 (582, np.float64(0.19654337143739709)),
 (1214, np.float64(0.19073064169416315)),
 (3606, np.float64(0.18120581984802397))]

In [106]:
recommend('Batman Begins')

The Dark Knight
The Dark Knight Rises
Batman
Batman v Superman: Dawn of Justice
Batman Returns


In [107]:
# import pickle
# pickle.dump(new_df, open('movies.pkl', 'wb'))

In [108]:
# pickle.dump(similarity, open('similarity.pkl', 'wb'))

## Version 2

In [109]:
# multiple movie handle function
def recommend_multple(movies):
    movies_vectors = []
    for movie in movies:
        index = int(new_df[new_df['title'] == movie].index[0])
        vector = vectors[index]
        movies_vectors.append(vector)
    user_vector = np.mean(movies_vectors, axis=0)
    similarities = cosine_similarity([user_vector],vectors)

In [110]:
import pickle
pickle.dump(new_df, open('movies.pkl', 'wb'))
pickle.dump(vectors, open('vectors.pkl', 'wb'))